# 03 — LoRA (rank 8)

**Goal:** the same task, the same data, the same 3 epochs — but only the LoRA adapters train. Everything else is frozen.

**LoRA config (this study):** rank `r=8`, `lora_alpha = 2r = 16`, dropout 0.1, target modules `q_lin` and `v_lin` (DistilBERT's attention projections), task type sequence classification. Learning rate 1e-4 — higher than full fine-tuning's 2e-5, because only a few parameters are being trained and they start from zero (see notebook 01).

In [ ]:
# Setup: run this first in Colab (or skip locally if requirements are installed)
# In Colab, clone the repo so src/ and experiments/ are available:
#   !git clone https://github.com/sujalgiriiitp-source/Lora---reproduction-.git
#   %cd Lora---reproduction-
# Then install dependencies (takes a few minutes the first time):
#   !pip install -q -r requirements.txt
import sys
print(sys.version)
try:
    import torch, transformers, peft, datasets
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
    print("transformers", transformers.__version__, "| peft", peft.__version__, "| datasets", datasets.__version__)
except ImportError as e:
    print("Missing package:", e)
    print("Run: pip install -r requirements.txt")


## 1. What gets trained? (the key check)

Before training, print the trainable parameters. If this number is not a small fraction of ~66M, the LoRA wrapping is wrong — stop here, do not train.

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, TaskType, get_peft_model

base = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.1,
                    target_modules=["q_lin", "v_lin"], task_type=TaskType.SEQ_CLS)
model = get_peft_model(base, config)
model.print_trainable_parameters()  # expect ~0.2M trainable of ~66M total

## 2. Train — LoRA r=8

Same entry point as the baseline (`src/train.py`), only `--mode` and `--r` differ. Same data, same seed, same epochs.

In [ ]:
# Full run (GPU). CPU smoke test: add --epochs 1 --max-train 200 --max-eval 100 --batch-size 8
!python src/train.py --mode lora --r 8 --epochs 3 --batch-size 32 --output runs/lora-r8 --run-id lora-r8

## 3. Compare with the baseline

Two rows, same table. The question is not just "did LoRA win" — it is "how much accuracy did we trade for how many fewer parameters, and how much faster was it?"

In [ ]:
import pandas as pd
runs = pd.read_csv("experiments/runs.csv")
cols = ["run_id", "mode", "lora_r", "trainable_params", "accuracy", "f1_macro", "train_time_sec"]
print(runs[cols].to_string(index=False))
base = runs.set_index("run_id").loc["full"]
lora = runs.set_index("run_id").loc["lora-r8"]
print(f"\nAccuracy gap (full - LoRA r=8): {base['accuracy'] - lora['accuracy']:.4f}")
print(f"Trainable params: LoRA uses {lora['trainable_params'] / base['trainable_params'] * 100:.2f}% of full fine-tuning")

**What to note:** if LoRA r=8 is within ~1 point of full fine-tuning, that *is* the paper's claim reproducing at small scale. If it is several points behind, say so — and check first whether the learning rate or the number of epochs is the cause before blaming the method.

**Next:** notebook 04 runs the other two ranks and turns the three LoRA runs + baseline into the ablation plots.